<a href="https://colab.research.google.com/github/Lukas-Swc/neural-network-course/blob/main/07_rnn/02_text_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import os

import tensorflow as tf

In [2]:
!wget https://storage.googleapis.com/esmartdata-courses-files/ann-course/reviews.zip
!unzip -q reviews.zip

--2025-07-01 16:28:58--  https://storage.googleapis.com/esmartdata-courses-files/ann-course/reviews.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.16.207, 192.178.155.207, 172.253.62.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.16.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42878657 (41M) [application/x-zip-compressed]
Saving to: ‘reviews.zip’

reviews.zip         100%[===================>]  40.89M  23.0MB/s    in 1.8s    

2025-07-01 16:29:00 (23.0 MB/s) - ‘reviews.zip’ saved [42878657/42878657]



In [3]:
data_dir = './reviews'
train_dir = os.path.join(data_dir, 'train')

train_texts = []
train_labels = []

for label_type in ['neg', 'pos']:
  dir_name = os.path.join(train_dir, label_type)
  for fname in os.listdir(dir_name):
    if fname[-4:] == '.txt':
      f = open(os.path.join(dir_name, fname))
      train_texts.append(f.read())
      f.close()
      if label_type == 'neg':
        train_labels.append(0)
      else:
        train_labels.append(1)

In [4]:
test_dir = os.path.join(data_dir, 'test')

test_texts = []
test_labels = []

for label_type in ['neg', 'pos']:
  dir_name = os.path.join(test_dir, label_type)
  for fname in os.listdir(dir_name):
    if fname[-4:] == '.txt':
      f = open(os.path.join(dir_name, fname))
      test_texts.append(f.read())
      f.close()
      if label_type == 'neg':
        test_labels.append(0)
      else:
        test_labels.append(1)

In [5]:
train_texts[:10]

["In The Lost Son, a private eye searching for a missing man stumbles upon a child prostitution ring. This film incorporates all of the worst stereotypes you could imagine in a worst-case scenario that exists only in the minds of Hollywood, the press and AG John Asscrap. If you get a chance to see this, you'd be better off getting lost yourself.",
 "haha! you have to just smile and smile if you actually made it all the way through this movie. it like says something about myself i guess. the movie itself was created i think as some sort of psychological test, or like some sort of drug, to take you to a place you have never been before. When Wittgenstein wrote his famous first philosophical piece the tractacus (sp?) he said it was meaningless and useless, but if you read it, after you were done, it would take you to a new level, like a ladder, and then you could throw away the work and see things with clarity and true understanding. this movie is the same i think.<br /><br />As a movie i

In [6]:
train_labels[:10]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [7]:
train_labels[-10:]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [8]:
maxlen = 100 # skracamy recenzje do 100 slow
num_words = 10000 # 10000 najczesciej pojawiajacych sie slow
embedding_dim = 100

tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=num_words)
tokenizer.fit_on_texts(train_texts)

In [9]:
list(tokenizer.index_word.items())[:20]

[(1, 'the'),
 (2, 'and'),
 (3, 'a'),
 (4, 'of'),
 (5, 'to'),
 (6, 'is'),
 (7, 'br'),
 (8, 'in'),
 (9, 'it'),
 (10, 'i'),
 (11, 'this'),
 (12, 'that'),
 (13, 'was'),
 (14, 'as'),
 (15, 'for'),
 (16, 'with'),
 (17, 'movie'),
 (18, 'but'),
 (19, 'film'),
 (20, 'on')]

In [10]:
sequences = tokenizer.texts_to_sequences(train_texts)
print(sequences[:3])

[[8, 1, 413, 489, 3, 1951, 741, 3159, 15, 3, 1009, 129, 5844, 722, 3, 503, 8169, 1740, 11, 19, 29, 4, 1, 246, 2103, 22, 97, 835, 8, 3, 246, 417, 2665, 12, 2974, 61, 8, 1, 2633, 4, 359, 1, 3513, 2, 304, 44, 22, 76, 3, 576, 5, 64, 11, 1383, 27, 125, 122, 394, 413, 620], [9457, 22, 25, 5, 40, 1822, 2, 1822, 44, 22, 162, 90, 9, 29, 1, 93, 140, 11, 17, 9, 37, 555, 139, 41, 543, 10, 479, 1, 17, 407, 13, 1072, 10, 101, 14, 46, 429, 4, 1983, 2178, 39, 37, 46, 429, 4, 1389, 5, 190, 22, 5, 3, 270, 22, 25, 112, 74, 156, 51, 1037, 24, 800, 83, 4311, 415, 1, 26, 298, 9, 13, 4010, 2, 3493, 18, 44, 22, 329, 9, 100, 22, 68, 221, 9, 59, 190, 22, 5, 3, 159, 646, 37, 3, 5399, 2, 92, 22, 97, 1395, 242, 1, 154, 2, 64, 180, 16, 7799, 2, 280, 1893, 11, 17, 6, 1, 169, 10, 101, 7, 7, 14, 3, 17, 9, 6, 206, 3, 821, 1, 246, 17, 10, 25, 107, 8, 3, 193, 193, 55, 8, 138, 3, 951, 93, 83, 4, 29, 11, 6, 7476, 10, 444, 146, 11, 229, 1965, 1992, 8, 995, 99, 2, 10, 25, 3108, 140, 3, 168, 812, 659, 187, 258, 22, 121, 1, 17

In [11]:
word_index = tokenizer.word_index
print(f'{len(word_index)} unikatowych slow')

88582 unikatowych slow


In [12]:
# skracamy recenzje do pierwszych 100 slow
train_data = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=maxlen)
train_data.shape

(25000, 100)

In [13]:
train_data[:3]

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    8,    1,  413,  489,
           3, 1951,  741, 3159,   15,    3, 1009,  129, 5844,  722,    3,
         503, 8169, 1740,   11,   19,   29,    4,    1,  246, 2103,   22,
          97,  835,    8,    3,  246,  417, 2665,   12, 2974,   61,    8,
           1, 2633,    4,  359,    1, 3513,    2,  304,   44,   22,   76,
           3,  576,    5,   64,   11, 1383,   27,  125,  122,  394,  413,
         620],
       [1095,  118,   22,   63,   89,   64,   48,  571,    2,   20,  347,
           4,   12, 1622,  390,    9,   22,   89,  456,   22,   68,  498,
        1096,   93,   36,    1,  451,    7,    7,   35,   10,    8,    2,
          43,    3,  375,  208,   18,   10, 1056,  217,   29,    4,   11,
          17,    2,    

In [14]:
train_labels = np.asarray(train_labels)
train_labels

array([0, 0, 0, ..., 1, 1, 1])

In [15]:
# przemieszanie probek
indices = np.arange(train_data.shape[0])
np.random.shuffle(indices)
train_data = train_data[indices]
train_labels = train_labels[indices]

train_data.shape

(25000, 100)

In [16]:
# podzial na zbior treningowy i walidacyjny
training_samples = 15000
validation_samples = 10000

X_train = train_data[:training_samples]
y_train = train_labels[:training_samples]
X_val = train_data[training_samples : training_samples + validation_samples]
y_val = train_labels[training_samples : training_samples + validation_samples]

In [17]:
# budowa modelu
# Embedding(input_dim, outpu_dim)

model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Embedding(input_dim=num_words, output_dim=embedding_dim))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(16, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

model.build(input_shape=(None, maxlen))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       160,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,160,033 (4.43 MB)

 Trainable params: 1,160,033 (4.43 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [19]:
history = model.fit(X_train, y_train, batch_size=32, epochs=5, validation_data=(X_val, y_val))

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.6630 - loss: 0.5903 - val_accuracy: 0.8257 - val_loss: 0.3833
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9298 - loss: 0.1972 - val_accuracy: 0.8115 - val_loss: 0.4407
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9926 - loss: 0.0349 - val_accuracy: 0.8184 - val_loss: 0.5405
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.9994 - loss: 0.0048 - val_accuracy: 0.8157 - val_loss: 0.6700
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9998 - loss: 6.1382e-04 - val_accuracy: 0.8130 - val_loss: 0.7573


In [20]:
def plot_hist(history):
    import pandas as pd
    import plotly.graph_objects as go
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['accuracy'], name='accuracy', mode='markers+lines'))
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['val_accuracy'], name='val_accuracy', mode='markers+lines'))
    fig.update_layout(width=1000, height=500, title='accuracy vs. val accuracy', xaxis_title='Epoki', yaxis_title='accuracy', yaxis_type='log')
    fig.show()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['loss'], name='loss', mode='markers+lines'))
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['val_loss'], name='val_loss', mode='markers+lines'))
    fig.update_layout(width=1000, height=500, title='loss vs. val loss', xaxis_title='Epoki', yaxis_title='loss', yaxis_type='log')
    fig.show()

plot_hist(history)

In [ ]:
sequences = tokenizer.texts_to_sequences(test_texts)
X_test = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=maxlen)
y_test = np.asarray(test_labels)

model.evaluate(X_test, y_test, verbose=0)

[0.7142333388328552, 0.8140400052070618]

### Simple RNN

In [26]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Embedding(10000, 32))
model.add(tf.keras.layers.SimpleRNN(16))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

model.build(input_shape=(None, None))

model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, None, 32)       │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ (None, 16)             │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,801 (1.22 MB)

 Trainable params: 320,801 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [28]:
history = model.fit(X_train, y_train, batch_size=32, epochs=10, validation_data=(X_val, y_val))

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.6801 - loss: 0.5806 - val_accuracy: 0.8274 - val_loss: 0.3982
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 30ms/step - accuracy: 0.8667 - loss: 0.3314 - val_accuracy: 0.8242 - val_loss: 0.3977
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 21s 31ms/step - accuracy: 0.9006 - loss: 0.2557 - val_accuracy: 0.8273 - val_loss: 0.4246
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - accuracy: 0.9287 - loss: 0.1923 - val_accuracy: 0.8258 - val_loss: 0.4056
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 20s 30ms/step - accuracy: 0.9486 - loss: 0.1444 - val_accuracy: 0.8398 - val_loss: 0.4556
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - accuracy: 0.9710 - loss: 0.0969 - val_accuracy: 0.8313 - val_loss: 0.4687
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.9767 - loss: 0.0689 - val_accuracy: 0.8208 - val_loss: 0.5330
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 22s 32ms/step - accuracy: 0.9882 - loss: 0.0438 - 

In [29]:
plot_hist(history)

In [30]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Embedding(10000, 32))
model.add(tf.keras.layers.LSTM(16))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

model.build(input_shape=(None, None))

model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, None, 32)       │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 323,153 (1.23 MB)

 Trainable params: 323,153 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [32]:
history = model.fit(X_train, y_train, batch_size=32, epochs=10, validation_data=(X_val, y_val))

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 29s 51ms/step - accuracy: 0.6546 - loss: 0.6042 - val_accuracy: 0.8281 - val_loss: 0.3942
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - accuracy: 0.8735 - loss: 0.3184 - val_accuracy: 0.8426 - val_loss: 0.3656
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 43s 50ms/step - accuracy: 0.9000 - loss: 0.2557 - val_accuracy: 0.8307 - val_loss: 0.3913
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 40s 47ms/step - accuracy: 0.9197 - loss: 0.2181 - val_accuracy: 0.8384 - val_loss: 0.4046
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 22s 46ms/step - accuracy: 0.9322 - loss: 0.1924 - val_accuracy: 0.8392 - val_loss: 0.3927
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 45s 55ms/step - accuracy: 0.9404 - loss: 0.1668 - val_accuracy: 0.8440 - val_loss: 0.4166
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 38s 49ms/step - accuracy: 0.9451 - loss: 0.1569 - val_accuracy: 0.8377 - val_loss: 0.4454
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 39s 46ms/step - accuracy: 0.9518 - loss: 0.1442 - 

In [33]:
plot_hist(history)

In [34]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Embedding(10000, 32))
model.add(tf.keras.layers.LSTM(16))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

model.build(input_shape=(None, None))

model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, None, 32)       │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 323,153 (1.23 MB)

 Trainable params: 323,153 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [37]:
history = model.fit(X_train, y_train, batch_size=32, epochs=3, validation_data=(X_val, y_val))

Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 28s 54ms/step - accuracy: 0.6392 - loss: 0.6200 - val_accuracy: 0.8134 - val_loss: 0.4196
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 40s 52ms/step - accuracy: 0.8629 - loss: 0.3364 - val_accuracy: 0.8242 - val_loss: 0.4452
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 37s 44ms/step - accuracy: 0.8996 - loss: 0.2589 - val_accuracy: 0.8175 - val_loss: 0.4442


In [38]:
sequences = tokenizer.texts_to_sequences(test_texts)
X_test = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=maxlen)
y_test = np.asarray(test_labels)

model.evaluate(X_test, y_test, verbose=0)

[0.4348563551902771, 0.8145999908447266]